In [2]:
import os
import pandas as pd
import psycopg

from pathlib import Path
from dotenv import load_dotenv
from psycopg.rows import dict_row

load_dotenv("../.env")

conn_params = {
    "host": "127.0.0.1",
    "port": 15432,
    "dbname": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"]
}

In [3]:
sql_files = {
    "monthly_conversion": "14_sessionized_conversion.sql",
    "conversion_funnel": "15_sessionized_funnel.sql",
    "weekly_conversion": "16_sessionized_weekly.sql",
    "purchase_segments": "17_sessionized_segments.sql"
}

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {}

with psycopg.connect(**conn_params, row_factory=dict_row) as conn:
    for name, filename in sql_files.items():
        query = Path(f"../sql/analysis/{filename}").read_text(encoding="utf-8")

        result = conn.execute(query).fetchall()
        df = pd.DataFrame(result)

        datasets[name] = df
        df.to_csv(output_dir / f"{name}.csv", index=False)

        print(f"{name}: {len(df)} rows")

monthly_conversion: 2 rows
conversion_funnel: 2 rows
weekly_conversion: 6 rows
purchase_segments: 4 rows


In [4]:
for name, df in datasets.items():
    print(f"\n{name}")
    display(df)


monthly_conversion


,month,view_sessions,converted_sessions,session_cr
0,November,637104,17988,2.82
1,October,573981,14075,2.45



conversion_funnel


,month,view_sessions,cart_sessions,purchase_sessions,view_to_cart_cr,cart_to_purchase_cr,full_funnel_cr
0,November,637104,111590,15605,17.52,13.98,2.45
1,October,573981,102667,12310,17.89,11.99,2.14



weekly_conversion


,week_start,view_sessions,converted_sessions,session_cr
0,2019-10-07,202832,4767,2.35
1,2019-10-14,184654,4677,2.53
2,2019-10-21,186495,4631,2.48
3,2019-11-04,200945,5377,2.68
4,2019-11-11,188429,4736,2.51
5,2019-11-18,247730,7875,3.18



purchase_segments


,month,segment,view_sessions,converted_sessions,session_cr
0,November,No prior purchase,538798,13201,2.45
1,November,Prior purchaser,98306,4787,4.87
2,October,No prior purchase,526096,12150,2.31
3,October,Prior purchaser,47885,1925,4.02
